# 利用多层全连接神经网络及简单卷积神经网络对MNIST数据集分类

导入相关包

In [ ]:
import torch
import torchvision as tv
import torchvision.transforms as transforms
import torch.nn as nn

定义是否使用GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


定义多层全连接神经网络

In [ ]:
class multlayer_network(nn.Module):
    '''
        全连接神经网络
    '''


定义LeNet神经网络

In [ ]:
class LeNet(nn.Module):
    '''
        LeNet神经网络
    '''

超参数设置

In [ ]:
EPOCH = 10  # 遍历数据集次数
BATCH_SIZE = 128  # 批处理尺寸(batch_size)
LR = 0.001  # 学习率

定义数据预处理方式

In [ ]:
transform = transforms.ToTensor()
trainset = tv.datasets.MNIST(
    root='/data/',
    train=True,
    download=True,  # 首次设定True，后面可以设置为False
    transform=transform)

# 定义训练批处理数据
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    )

# 定义测试数据集
testset = tv.datasets.MNIST(
    root='/data/',
    train=False,
    download=True,
    transform=transform)

# 定义测试批处理数据
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    )

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 480kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.96MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.1MB/s]


实例化网络，如有GPU，将模型加载至GPU运算

In [ ]:
# ====================== 1. 多层全连接神经网络 (MLP) ======================
class MLP(nn.Module):
  def __init__(self):
    super(MLP,self).__init__()
    self.flatten = nn.Flatten()
    self.net = nn.Sequential(
        nn.Linear(28 * 28,512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(512,256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(256,128),
        nn.ReLU(inplace=True),
        nn.Linear(128,10)
    )
  def forward(self,x):
    x = self.flatten(x)
    return self.net(x)



In [ ]:
# ====================== 2. 简单卷积神经网络 (CNN) ======================
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN,self).__init__()
    self.features = nn.Sequential(
        nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2),

    )
    self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128,10)
        )
  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [ ]:
model = MLP()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

定义损失函数和优化方式，尝试SGD、Adam等优化器，尝试运用权重衰减

In [ ]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
#1:Adam + 权重衰减
optimizer_adam = optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)


In [ ]:
#2:SGD + Momentum + 权重衰减
optimizer_sgd_momentum = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay = 5e-4
)

In [ ]:
#3:AdamW
optimizer_adamw = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

In [ ]:
optimizers = {
    "SGD + Momentum": optimizer_sgd_momentum,
    "Adam": optimizer_adam,
    "AdamW": optimizer_adamw
}

模型训练及测试

In [ ]:
#加载MNIST数据集
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"数据加载完成！训练集大小: {len(train_dataset)}, 测试集大小: {len(test_dataset)}")

数据加载完成！训练集大小: 60000, 测试集大小: 10000


In [ ]:
# ====================== 训练函数（支持不同模型和优化器） ======================
def train_model(model, optimizer, criterion, train_loader, test_loader, device, EPOCH, model_name, opt_name):
    print(f"\n开始训练 → 模型: {model_name} | 优化器: {opt_name}")
    print("-" * 70)

    best_acc = 0.0

    for epoch in range(EPOCH):
        # ------------------ 训练阶段 ------------------
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        train_acc = 100. * correct_train / total_train
        avg_train_loss = train_loss / len(train_loader)

        # ------------------ 测试阶段 ------------------
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                test_loss += loss.item()

                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_acc = 100. * correct / total
        avg_test_loss = test_loss / len(test_loader)

        # 打印每轮结果
        print(f"Epoch [{epoch+1:2d}/{EPOCH}]  "
              f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.2f}%  |  "
              f"Test Loss: {avg_test_loss:.4f}  Acc: {test_acc:.2f}%")

        # 更新最佳准确率
        if test_acc > best_acc:
            best_acc = test_acc

    print(f"{model_name} + {opt_name} 训练完成！最佳测试准确率: {best_acc:.2f}%\n")
    return best_acc


# ====================== 执行训练（MLP 和 CNN 各用 3 个优化器） ======================

results = {}   # 用于保存所有实验结果

# 要训练的模型组合
models_to_train = {
    "MLP": MLP,
    "CNN": SimpleCNN
}

optimizers_dict = {
    "SGD + Momentum": optimizer_sgd_momentum,
    "Adam": optimizer_adam,
    "AdamW": optimizer_adamw
}

# 开始训练所有组合
for model_name, model_class in models_to_train.items():
    for opt_name, optimizer in optimizers_dict.items():

        # 每次都重新实例化模型（保证实验公平）
        model = model_class().to(device)

        # 关键：为当前模型 + 优化器创建一个新的优化器实例
        if opt_name == "SGD + Momentum":
            current_optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
        elif opt_name == "Adam":
            current_optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0)
        else:  # AdamW
            current_optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

        # 开始训练
        best_acc = train_model(
            model=model,
            optimizer=current_optimizer,
            criterion=criterion,
            train_loader=train_loader,
            test_loader=test_loader,
            device=device,
            EPOCH=EPOCH,
            model_name=model_name,
            opt_name=opt_name
        )

        # 保存结果
        key = f"{model_name} + {opt_name}"
        results[key] = best_acc

# ====================== 最终结果对比 ======================
print("=" * 80)
print("所有实验完成！最终结果汇总：")
print("=" * 80)

for combo, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{combo:35s} → 最佳测试准确率: {acc:.2f}%")

print("=" * 80)# ====================== 训练函数（支持不同模型和优化器） ======================
def train_model(model, optimizer, criterion, train_loader, test_loader, device, EPOCH, model_name, opt_name):
    print(f"\n开始训练 → 模型: {model_name} | 优化器: {opt_name}")
    print("-" * 70)

    best_acc = 0.0

    for epoch in range(EPOCH):
        # ------------------ 训练阶段 ------------------
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        train_acc = 100. * correct_train / total_train
        avg_train_loss = train_loss / len(train_loader)

        # ------------------ 测试阶段 ------------------
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                test_loss += loss.item()

                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_acc = 100. * correct / total
        avg_test_loss = test_loss / len(test_loader)

        # 打印每轮结果
        print(f"Epoch [{epoch+1:2d}/{EPOCH}]  "
              f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.2f}%  |  "
              f"Test Loss: {avg_test_loss:.4f}  Acc: {test_acc:.2f}%")

        # 更新最佳准确率
        if test_acc > best_acc:
            best_acc = test_acc

    print(f"{model_name} + {opt_name} 训练完成！最佳测试准确率: {best_acc:.2f}%\n")
    return best_acc


# ====================== 执行训练（MLP 和 CNN 各用 3 个优化器） ======================

results = {}   # 用于保存所有实验结果

# 要训练的模型组合
models_to_train = {
    "MLP": MLP,
    "CNN": SimpleCNN
}

optimizers_dict = {
    "SGD + Momentum": optimizer_sgd_momentum,
    "Adam": optimizer_adam,
    "AdamW": optimizer_adamw
}

# 开始训练所有组合
for model_name, model_class in models_to_train.items():
    for opt_name, optimizer in optimizers_dict.items():

        # 每次都重新实例化模型（保证实验公平）
        model = model_class().to(device)

        # 关键：为当前模型 + 优化器创建一个新的优化器实例
        if opt_name == "SGD + Momentum":
            current_optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
        elif opt_name == "Adam":
            current_optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0)
        else:  # AdamW
            current_optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

        # 开始训练
        best_acc = train_model(
            model=model,
            optimizer=current_optimizer,
            criterion=criterion,
            train_loader=train_loader,
            test_loader=test_loader,
            device=device,
            EPOCH=EPOCH,
            model_name=model_name,
            opt_name=opt_name
        )

        # 保存结果
        key = f"{model_name} + {opt_name}"
        results[key] = best_acc

# ====================== 最终结果对比 ======================
print("=" * 80)
print("所有实验完成！最终结果汇总：")
print("=" * 80)

for combo, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{combo:35s} → 最佳测试准确率: {acc:.2f}%")

print("=" * 80)


开始训练 → 模型: MLP | 优化器: SGD + Momentum
----------------------------------------------------------------------
Epoch [ 1/10]  Train Loss: 0.5886  Acc: 82.33%  |  Test Loss: 0.1742  Acc: 94.63%
Epoch [ 2/10]  Train Loss: 0.1677  Acc: 94.97%  |  Test Loss: 0.1098  Acc: 96.49%
Epoch [ 3/10]  Train Loss: 0.1164  Acc: 96.47%  |  Test Loss: 0.0862  Acc: 97.35%
Epoch [ 4/10]  Train Loss: 0.0878  Acc: 97.30%  |  Test Loss: 0.0744  Acc: 97.66%
Epoch [ 5/10]  Train Loss: 0.0717  Acc: 97.82%  |  Test Loss: 0.0682  Acc: 97.80%
Epoch [ 6/10]  Train Loss: 0.0610  Acc: 98.14%  |  Test Loss: 0.0680  Acc: 97.73%
Epoch [ 7/10]  Train Loss: 0.0539  Acc: 98.33%  |  Test Loss: 0.0622  Acc: 97.92%
Epoch [ 8/10]  Train Loss: 0.0443  Acc: 98.67%  |  Test Loss: 0.0619  Acc: 98.04%
Epoch [ 9/10]  Train Loss: 0.0413  Acc: 98.73%  |  Test Loss: 0.0607  Acc: 98.06%
Epoch [10/10]  Train Loss: 0.0370  Acc: 98.86%  |  Test Loss: 0.0600  Acc: 98.13%
MLP + SGD + Momentum 训练完成！最佳测试准确率: 98.13%


开始训练 → 模型: MLP | 优化器: Adam
